# MUSANGA V3.1 — NB07: Independent Luki UAV validation

Use the **frozen final M2 warm-start ResNet18 U-Net**, exactly as in NB02B and NB03, on the new orthomosaic `Luki.tif`. **No Luki training, no fine-tuning, no test-set threshold fitting.**

**Primary reference:** each polygon in `Circles_Luki_Annotations.shp` is a positive Musanga tree crown. Evaluate all valid RGB pixels *inside* `Circles_Luki_ROI.shp`; the remaining valid pixels in those exhaustive circles are known negatives. Polygon/circle intersection (not centroid ownership) defines truth.

**Secondary reference:** `Tree_Annotation_Luki_Nadine.shp`, `Class_ID=1`, for visual comparison against the full-mosaic predictions. This is not substituted for exhaustive circle truth.

**Workflow:** 00 inputs/provenance → 01 geometry/CRS QA → 02 frozen model → 03 native-resolution circle inference → 04 external evaluation → 05 figures → 06 optional full-mosaic inference → 07 optional Nadine comparison. Outputs are separate in `Outputs/V3_1/07_Luki_external_validation`. Full-mosaic inference is **off by default**.

The model was trained on the original non-Luki circles; this notebook checks the saved training/input manifest for Luki references. The independent evaluation is contingent on that provenance being correct.

## 00 — Drive, audited model, and Luki data
NB07 revised: annotation paths use Data/Annotations; invalid Nadine polygons are validated and repaired in memory (area change ≤1%). Frozen warm-start model and validation methodology unchanged.


In [ ]:
%pip -q install rasterio geopandas shapely segmentation-models-pytorch scikit-learn matplotlib tqdm
from pathlib import Path
import os,json,hashlib,math,time,shutil
import numpy as np,pandas as pd,geopandas as gpd,rasterio,torch,matplotlib.pyplot as plt
from rasterio.windows import Window,from_bounds
from rasterio.features import rasterize
from shapely.geometry import box
from shapely.ops import unary_union
from sklearn.metrics import r2_score
from IPython.display import display
from google.colab import drive
from tqdm.auto import tqdm
if not Path('/content/drive/MyDrive').exists():drive.mount('/content/drive')
ROOT=next((p for p in (Path('/content/drive/MyDrive/Colab Notebooks/Musanga/Final'),
                         Path('/content/drive/MyDrive/Musanga/Final')) if p.is_dir()),None)
assert ROOT is not None,'Musanga Final workspace not found'
DATA=ROOT/'Data'
ANNOT=DATA/'Annotations'
ORTHO=DATA/'Ortho_RGB' if (DATA/'Ortho_RGB').is_dir() else ROOT/'Ortho_RGB'
def exact_one(folder,names):
    hits=[folder/n for n in names if (folder/n).is_file()]
    assert len(hits)==1,('Expected exactly one file',str(folder),names,[str(x) for x in hits])
    return hits[0]
RGB=exact_one(ORTHO,['Luki.tif','Luki_georef.tif'])
CIRCLES=exact_one(ANNOT,['Circles_Luki_ROI.shp'])
CROWNS=exact_one(ANNOT,['Circles_Luki_Annotations.shp','Circles_LukiAnnotation.shp'])
NADINE=exact_one(ANNOT,['Tree_Annotations_Luki_Nadine.shp','Tree_Annotation_Luki_Nadine.shp'])
BASE=ROOT/'Outputs/V3_1/02B_spatial3fold_no_MOS34'
FINAL=BASE/'final_warmstart_M2_c6a33504c37efb2ebb84'
CHECKPOINT=FINAL/'M2_warmstart_all30_epoch02.pt'
MANIFEST=FINAL/'manifest.json'
AUDIT=ROOT/'Outputs/V3_1/00_input_audit/00_input_manifest.json'
assert all(p.is_file() for p in (CHECKPOINT,MANIFEST,AUDIT))
cfg=json.loads(MANIFEST.read_text());audit=json.loads(AUDIT.read_text())
assert cfg.get('version')=='M2_V3_1_warmstart_cv0_to_all30_v1'
assert cfg.get('crop')==256 and cfg.get('epochs')==2
assert 'luki' not in json.dumps(cfg).lower()
assert 'luki' not in json.dumps(audit.get('mosaics',[])).lower(), 'Luki present in original mosaic inventory'
assert torch.cuda.is_available(),'Select Colab GPU'
def sha_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for a in iter(lambda:f.read(8*1024*1024),b''):h.update(a)
    return h.hexdigest()
MODEL_SHA=sha_file(CHECKPOINT)
OUT=ROOT/'Outputs/V3_1/07_Luki_external_validation'
OUT.mkdir(parents=True,exist_ok=True)
contract=dict(rgb=str(RGB),circles=str(CIRCLES),crowns=str(CROWNS),
              nadine=str(NADINE),checkpoint=str(CHECKPOINT),checkpoint_sha256=MODEL_SHA)
lock=OUT/'07_input_contract.json'
if lock.exists(): assert json.loads(lock.read_text())==contract,'Source paths or checkpoint changed; review cache'
else:lock.write_text(json.dumps(contract,indent=2))
print('Frozen checkpoint:',CHECKPOINT,'\nSHA256:',MODEL_SHA)
print('Luki imagery:',RGB,'\nEvaluation:',CIRCLES,'\nCrowns:',CROWNS,'\nNadine:',NADINE)

## 01 — Validate CRS, exhaustive polygons, and Luki RGB compatibility

In [ ]:
with rasterio.open(RGB) as ds:
    assert ds.crs is not None and ds.count>=3
    assert all(d=='uint8' for d in ds.dtypes[:3]),('NB03 requires native uint8 RGB',ds.dtypes)
    assert ds.width>256 and ds.height>256
    CRS=ds.crs;BOUNDS=box(*ds.bounds)
    print('Dimensions',ds.width,ds.height,'resolution',ds.res,'CRS',ds.crs,'bands',ds.count)
cir=gpd.read_file(CIRCLES);crown=gpd.read_file(CROWNS);nad=gpd.read_file(NADINE)
for name,g in [('circles',cir),('exhaustive crowns',crown),('Nadine',nad)]:
    assert g.crs is not None,(name,'Missing CRS; no automatic guessing')
    assert len(g)>0 and g.geometry.notna().all()
    if name!='Nadine':
        assert g.geometry.is_valid.all(),(name,'Invalid geometry; repair source and rerun')
    assert g.geom_type.isin(['Polygon','MultiPolygon']).all(),(name,'expected polygons')
# Nadine has two topologically invalid digitized polygons. Repair only these
# in memory, preserving the original source file and polygon count.
from shapely.validation import explain_validity
try:
    from shapely import make_valid
except ImportError:
    from shapely.validation import make_valid
invalid_nad=~nad.geometry.is_valid
print('Nadine invalid polygon count:',int(invalid_nad.sum()))
for idx in nad.index[invalid_nad]:
    original=nad.at[idx,'geometry']
    fixed=make_valid(original)
    # Do not silently discard lines, points or disconnected pieces.
    assert fixed.geom_type in ('Polygon','MultiPolygon'), (
        'Nadine repair requires manual inspection',idx,
        explain_validity(original),fixed.geom_type)
    assert original.area>0,('Nadine original area is zero',idx)
    delta=abs(fixed.area-original.area)/original.area
    assert delta<=0.01,('Nadine repair area change exceeds 1%',idx,delta)
    assert fixed.is_valid,('Nadine repair still invalid',idx)
    nad.at[idx,'geometry']=fixed
assert nad.geometry.is_valid.all()
assert len(nad)>0 and nad.geom_type.isin(['Polygon','MultiPolygon']).all()
print('Nadine geometry QA passed; source shapefile unchanged')
cir=cir.to_crs(CRS).reset_index(drop=True)
crown=crown.to_crs(CRS).reset_index(drop=True)
nad=nad.to_crs(CRS).reset_index(drop=True)
assert 'Class_ID' in nad, 'Nadine Class_ID absent'
nad_pos=nad[pd.to_numeric(nad.Class_ID,errors='coerce').eq(1)].copy()
assert len(nad_pos)>0
assert all(g.intersects(BOUNDS) for g in cir.geometry),'Circle outside raster'
cir['eval_id']=np.arange(len(cir))
assert abs(sum(g.area for g in cir.geometry)-unary_union(cir.geometry.tolist()).area)<1e-6*sum(g.area for g in cir.geometry), 'Overlapping exhaustive circles; correct boundaries first'
print('Circles:',len(cir),'exhaustive positive crowns:',len(crown),'Nadine positives:',len(nad_pos))
print('Circle fields:',list(cir.columns),'Crown fields:',list(crown.columns))
display(pd.DataFrame([dict(circle=int(i),area_m2=float(g.area),crowns_intersecting=int(crown.intersects(g).sum()))
    for i,g in enumerate(cir.geometry)]))

## 02 — Exact production M2 and globally anchored, feathered native inference

Code below is copied from NB03's core inference routine. The **same single epoch-02 checkpoint** is used, with no model averaging beyond this one model.

In [ ]:
import segmentation_models_pytorch as smp
class ResNet18UNet(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.net=smp.Unet(encoder_name='resnet18',encoder_weights=None,in_channels=3,classes=1)
    def forward(self,x):return self.net(x)[:,0]
model=ResNet18UNet().cuda()
model.load_state_dict(torch.load(CHECKPOINT,map_location='cpu',weights_only=True),strict=True)
model.eval()
MODELS=[model]
TILE=256;STEP=128;CORE=512;STRIPE=1024;BATCH=12
SCALE=10000;NODATA=65535;EDGE_FLOOR=.05
# EXACT NB02B tiled-256 QA feather, including its positive edge floor.
u=np.arange(TILE,dtype="float32")
u=np.minimum(1.,np.maximum(0.05,np.minimum((u+.5)/32.,(TILE-u-.5)/32.)))
WEIGHT=np.outer(u,u).astype("float32")
assert WEIGHT.min()>0
def valid_rgb(raw):
    # MATCH NB02B: masked RGB bands, excluding exact all-black, all-white, blue.
    a=raw.filled(0).astype("float32")
    v=~np.any(np.ma.getmaskarray(raw),axis=0)
    v &= np.any(a!=0,axis=0)&np.any(a!=255,axis=0)
    v &= ~((a[0]==0)&(a[1]==0)&(a[2]==255))
    return v

def starts(lo,hi):
    return list(range((lo-TILE+1)//STEP*STEP,(hi-1)//STEP*STEP+1,STEP))

def infer_rect(src,r0,c0,h,w):
    """Globally anchored overlapping 256-pixel windows; no core-dependent seams."""
    assert 0<=r0<src.height and 0<=c0<src.width
    assert 0<h<=src.height-r0 and 0<w<=src.width-c0
    center=src.read([1,2,3],window=Window(c0,r0,w,h),masked=True)
    valid=valid_rgb(center)
    if not valid.any():
        return np.full((h,w),np.nan,"float32"),valid
    rs=starts(r0,r0+h);cs=starts(c0,c0+w)
    top=min(rs);left=min(cs);bottom=max(rs)+TILE;right=max(cs)+TILE
    ext=np.zeros((3,bottom-top,right-left),dtype="uint8")
    rr0=max(0,top);cc0=max(0,left)
    rr1=min(bottom,src.height);cc1=min(right,src.width)
    if rr1>rr0 and cc1>cc0:
        src_data=src.read([1,2,3],window=Window(cc0,rr0,cc1-cc0,rr1-rr0),masked=True)
        ext[:,rr0-top:rr1-top,cc0-left:cc1-left]=src_data.filled(0)
    total=np.zeros((h,w),dtype="float64")
    weights_total=np.zeros((h,w),dtype="float64")
    patches=[];tasks=[]
    def flush():
        if not patches:return
        batch=torch.from_numpy(np.stack(patches).astype("float32")/255.).cuda()
        ensemble=None
        with torch.inference_mode():
            for model in MODELS:
                # Exact NB02B evaluation: FP32; no autocast.
                prediction=torch.sigmoid(model(batch)).float()
                ensemble=prediction if ensemble is None else ensemble+prediction
            predictions=(ensemble/len(MODELS)).cpu().numpy()
        for (tr,tc,cr0,cr1,ccr0,ccr1),prob in zip(tasks,predictions):
            pr0=cr0-(tr-r0);pc0=ccr0-(tc-c0)
            hh=cr1-cr0;ww=ccr1-ccr0
            wwgt=WEIGHT[pr0:pr0+hh,pc0:pc0+ww]
            pp=prob[pr0:pr0+hh,pc0:pc0+ww]
            total[cr0:cr1,ccr0:ccr1]+=wwgt*pp
            weights_total[cr0:cr1,ccr0:ccr1]+=wwgt
        patches.clear();tasks.clear()
    for tr in rs:
        for tc in cs:
            cr0=max(0,tr-r0);cr1=min(h,tr+TILE-r0)
            ccr0=max(0,tc-c0);ccr1=min(w,tc+TILE-c0)
            if cr0>=cr1 or ccr0>=ccr1:continue
            if not valid[cr0:cr1,ccr0:ccr1].any():continue
            patch=ext[:,tr-top:tr-top+TILE,tc-left:tc-left+TILE]
            assert patch.shape==(3,TILE,TILE)
            patches.append(patch);tasks.append((tr,tc,cr0,cr1,ccr0,ccr1))
            if len(patches)>=BATCH:flush()
    flush()
    assert np.all(weights_total[valid]>0),("Missing tile weight",r0,c0)
    p=np.full((h,w),np.nan,"float32")
    p[valid]=(total[valid]/weights_total[valid]).astype("float32")
    assert np.all(np.isfinite(p[valid])) and np.all((p[valid]>=0)&(p[valid]<=1))
    return p,valid

def to_u16(prob,valid):
    arr=np.full(prob.shape,NODATA,dtype="uint16")
    arr[valid]=np.rint(prob[valid]*SCALE).astype("uint16")
    assert np.array_equal(arr!=NODATA,valid)
    return arr


## 03 — Independent Luki circle inference (GPU, checkpointed)

Run native 512×512 cores covering the *bounding windows* of each exhaustive ROI circle, with the **global tile lattice** from NB03. Never infer only inside crowns. Save only pixels in the circle bounding boxes; avoid full mosaic inference at this validation stage.

In [ ]:
CIRCLE_MAPS=OUT/'03_circle_probability';CIRCLE_MAPS.mkdir(exist_ok=True)
circle_registry=[]
def pixel_window(ds,geom):
    w=from_bounds(*geom.bounds,transform=ds.transform)
    c0=max(0,int(math.floor(w.col_off)));r0=max(0,int(math.floor(w.row_off)))
    c1=min(ds.width,int(math.ceil(w.col_off+w.width)))
    r1=min(ds.height,int(math.ceil(w.row_off+w.height)))
    assert c1>c0 and r1>r0
    return Window(c0,r0,c1-c0,r1-r0)
with rasterio.open(RGB) as src:
    for idx,geom in enumerate(cir.geometry):
        win=pixel_window(src,geom)
        outpath=CIRCLE_MAPS/f'Luki_circle_{idx:03d}_M2_prob_u16.tif'
        tags=dict(checkpoint_sha256=MODEL_SHA,circle_id=str(idx),
                  rgb=str(RGB),col=str(int(win.col_off)),row=str(int(win.row_off)),
                  width=str(int(win.width)),height=str(int(win.height)))
        if outpath.exists():
            with rasterio.open(outpath) as prior:
                assert prior.width==win.width and prior.height==win.height
                assert prior.crs==src.crs and prior.transform==src.window_transform(win)
                assert all(prior.tags().get(k)==v for k,v in tags.items()),'Stale circle cache: inspect, do not overwrite'
            print('CACHE HIT circle',idx,flush=True)
        else:
            profile=src.profile.copy()
            for key in ('photometric','alpha','extrasamples','NBITS','colorinterp','interleave'):profile.pop(key,None)
            profile.update(driver='GTiff',count=1,dtype='uint16',nodata=NODATA,
                height=int(win.height),width=int(win.width),transform=src.window_transform(win),
                tiled=True,blockxsize=256,blockysize=256,compress='deflate',
                predictor=2,bigtiff='IF_SAFER')
            tmp=outpath.with_suffix('.partial.tif')
            assert not tmp.exists(),('Incomplete temporary output, inspect:',tmp)
            with rasterio.open(tmp,'w',**profile) as dst:
                for dr in range(0,int(win.height),CORE):
                    h=min(CORE,int(win.height)-dr)
                    for dc in range(0,int(win.width),CORE):
                        w=min(CORE,int(win.width)-dc)
                        prob,valid=infer_rect(src,int(win.row_off)+dr,int(win.col_off)+dc,h,w)
                        dst.write(to_u16(prob,valid),1,window=Window(dc,dr,w,h))
                dst.update_tags(**tags,COMPLETE='YES')
            os.replace(tmp,outpath)
            print('SAVED circle',idx,outpath,flush=True)
        circle_registry.append(dict(circle_id=idx,path=str(outpath),
            col=int(win.col_off),row=int(win.row_off),width=int(win.width),height=int(win.height)))
registry=pd.DataFrame(circle_registry)
registry.to_csv(OUT/'03_circle_map_registry.csv',index=False)
display(registry)

## 04 — Primary external validation: exhaustive Luki circles

Rasterize **all intersecting Musanga crowns** within each circle. Restrict all TP, FP, FN and fractional cover calculations to circle ∩ valid RGB. Classification threshold **0.5 is fixed**, exactly as NB03. Export per-circle metrics and pooled pixel statistics. R² requires at least two circles with nonconstant observed cover.

In [ ]:
def burn_geoms(series,win,ds):
    bounds=box(*rasterio.windows.bounds(win,ds.transform))
    ss=[g for g in series if not g.is_empty and g.intersects(bounds)]
    return rasterize(((g,1) for g in ss),out_shape=(int(win.height),int(win.width)),
                     transform=ds.window_transform(win),fill=0,dtype='uint8').astype(bool)
records=[];TP=FP=FN=TN=0
with rasterio.open(RGB) as src:
    for rec in registry.itertuples(index=False):
        idx=int(rec.circle_id);win=Window(rec.col,rec.row,rec.width,rec.height)
        with rasterio.open(rec.path) as dst:
            raw=src.read([1,2,3],window=win,masked=True)
            valid=valid_rgb(raw)
            circle_mask=burn_geoms([cir.geometry.iloc[idx]],win,src)
            evaluation=circle_mask&valid
            assert evaluation.any(),('No valid circle pixels',idx)
            truth=burn_geoms(crown.geometry,win,src)[evaluation]
            packed=dst.read(1)[evaluation]
            assert np.all(packed!=NODATA),('Prediction NoData within evaluation',idx)
            proba=packed.astype('float64')/SCALE
            hard=proba>=.5
            tp=int((hard&truth).sum());fp=int((hard&~truth).sum())
            fn=int((~hard&truth).sum());tn=int((~hard&~truth).sum())
            TP+=tp;FP+=fp;FN+=fn;TN+=tn
            observed=100*truth.mean();predicted=100*proba.mean()
            records.append(dict(circle_id=idx,n_pixels=len(truth),
                 observed_pct=observed,predicted_pct=predicted,
                 cover_bias_pp=predicted-observed,cover_abs_error_pp=abs(predicted-observed),
                 pixel_precision=tp/max(tp+fp,1),pixel_recall=tp/max(tp+fn,1),
                 pixel_F1=2*tp/max(2*tp+fp+fn,1),pixel_IoU=tp/max(tp+fp+fn,1),
                 TP=tp,FP=fp,FN=fn,TN=tn))
        print(f'Circle {idx:03d}: observed {observed:.2f}% predicted {predicted:.2f}% F1 {records[-1]["pixel_F1"]:.3f}',flush=True)
score=pd.DataFrame(records)
assert len(score)==len(cir)
o=score.observed_pct.to_numpy();p=score.predicted_pct.to_numpy()
summary=pd.DataFrame([dict(n_circles=len(score),mean_MAE_pp=float(np.abs(p-o).mean()),
    RMSE_pp=float(np.sqrt(np.mean((p-o)**2))),bias_pp=float((p-o).mean()),
    R2=float(r2_score(o,p)) if len(score)>1 and np.var(o)>0 else np.nan,
    pixel_precision=TP/max(TP+FP,1),pixel_recall=TP/max(TP+FN,1),
    pixel_F1=2*TP/max(2*TP+FP+FN,1),pixel_IoU=TP/max(TP+FP+FN,1),
    total_reference_pixels=int((TP+FP+FN+TN)))])
score.to_csv(OUT/'04_Luki_external_per_circle.csv',index=False)
summary.to_csv(OUT/'04_Luki_external_summary.csv',index=False)
print('=== EXTERNAL LUKI RESULTS ===');display(summary.round(3));display(score.round(3))

## 05 — External validation figures (no inference)

Circle scatterplot, error distribution, and selected RGB / reference / probability panels, all on exact overlapping pixels.

In [ ]:
fig,ax=plt.subplots(figsize=(6,6),layout='constrained')
ax.scatter(score.observed_pct,score.predicted_pct,s=55)
limit=max(10,float(np.maximum(score.observed_pct.max(),score.predicted_pct.max()))*1.1)
ax.plot([0,limit],[0,limit],'k--',lw=1)
for row in score.itertuples(index=False):
    ax.annotate(str(row.circle_id),(row.observed_pct,row.predicted_pct),xytext=(4,3),textcoords='offset points',fontsize=8)
ax.set(xlim=(0,limit),ylim=(0,limit),xlabel='Exhaustive annotated cover (%)',
       ylabel='Predicted Musanga probability mean (%)',
       title='Luki external validation — frozen M2')
ax.set_aspect('equal')
fig.savefig(OUT/'05_Luki_circle_scatter.png',dpi=180);plt.show();plt.close(fig)
with rasterio.open(RGB) as src:
    for idx in range(min(3,len(cir))):
        r=registry.iloc[idx];win=Window(int(r.col),int(r.row),int(r.width),int(r.height))
        rgb=src.read([1,2,3],window=win,masked=True).filled(0).transpose(1,2,0)
        ref=burn_geoms(crown.geometry,win,src)
        roi=burn_geoms([cir.geometry.iloc[idx]],win,src)
        with rasterio.open(r.path) as f:p=f.read(1).astype('float32')
        p[p==NODATA]=np.nan;p/=SCALE
        fig,axs=plt.subplots(1,3,figsize=(13,4),layout='constrained')
        axs[0].imshow(rgb);axs[0].set_title('Luki RGB')
        axs[1].imshow(np.ma.masked_where(~roi,ref),vmin=0,vmax=1);axs[1].set_title('Exhaustive crowns inside ROI')
        im=axs[2].imshow(np.ma.masked_where(~roi,p),vmin=0,vmax=1)
        axs[2].set_title('Frozen M2 probability');fig.colorbar(im,ax=axs[2],shrink=.7)
        for ax in axs:ax.axis('off')
        fig.savefig(OUT/f'05_Luki_circle_{idx:03d}_panels.png',dpi=160)
        plt.show();plt.close(fig)

## 06 — Optional entire Luki orthomosaic inference (GPU; **disabled by default**)

Use this only after validating circle results. Unlike circle windows, this writes a full-size output in 1024-row checkpointed stripes, then assembles a GeoTIFF with internal overviews. All native valid RGB is predicted, including outside annotated circles. Input/source signature guards prevent mixing checkpoints. Large Luki mosaics can require substantial disk space and GPU time.

In [ ]:
RUN_FULL_LUKI=False  # Explicitly switch to True only when full map is needed.
if not RUN_FULL_LUKI:
    print('Full orthomosaic inference disabled. Circle validation is complete without it.')
else:
    from rasterio.enums import Resampling
    FULL=OUT/'06_full_mosaic';PARTS=FULL/'stripes'
    PARTS.mkdir(parents=True,exist_ok=True)
    with rasterio.open(RGB) as src:
        file_stat=RGB.stat()
        spec=dict(width=src.width,height=src.height,crs=str(src.crs),transform=list(src.transform)[:6],
                  rgb=str(RGB),rgb_bytes=file_stat.st_size,rgb_mtime_ns=file_stat.st_mtime_ns,
                  checkpoint_sha256=MODEL_SHA,tile=TILE,step=STEP,stripe=STRIPE,
                  scale=SCALE,nodata=NODATA)
        key=hashlib.sha256(json.dumps(spec,sort_keys=True).encode()).hexdigest()[:20]
        d=PARTS/key;d.mkdir(parents=True,exist_ok=True)
        sf=d/'spec.json'
        if sf.exists():assert json.loads(sf.read_text())==spec
        else:sf.write_text(json.dumps(spec,indent=2))
        profile=src.profile.copy()
        for field in ('photometric','alpha','extrasamples','NBITS','colorinterp','interleave'):profile.pop(field,None)
        profile.update(driver='GTiff',count=1,dtype='uint16',nodata=NODATA,
                       tiled=True,blockxsize=256,blockysize=256,compress='deflate',
                       predictor=2,bigtiff='YES')
        fullpath=FULL/'Luki_M2_warmstart_epoch02_probability_u16_FULL.tif'
        def verify_stripe(path,row,h):
            if not path.exists():return False
            with rasterio.open(path) as f:
                return (f.width==src.width and f.height==h and f.dtypes[0]=='uint16'
                        and f.nodata==NODATA and f.tags().get('SPEC')==key
                        and f.tags().get('COMPLETE')=='YES' and
                        f.transform==src.window_transform(Window(0,row,src.width,h)))
        if fullpath.exists():
            with rasterio.open(fullpath) as f:
                assert f.tags().get('SPEC')==key and f.tags().get('COMPLETE')=='YES'
                assert f.shape==src.shape and f.transform==src.transform
            print('REUSED verified full Luki map:',fullpath)
        else:
            for k,row in enumerate(range(0,src.height,STRIPE)):
                h=min(STRIPE,src.height-row);dest=d/f'stripe_{k:05d}.tif'
                if verify_stripe(dest,row,h):
                    print('CACHE',k,flush=True);continue
                assert not dest.exists(),('Invalid stripe cache; inspect',dest)
                tmp=dest.with_suffix('.partial.tif')
                assert not tmp.exists(),('Incomplete temp stripe',tmp)
                p=profile.copy();p.update(height=h,transform=src.window_transform(Window(0,row,src.width,h)))
                with rasterio.open(tmp,'w',**p) as dst:
                    for dr in range(row,row+h,CORE):
                        hh=min(CORE,row+h-dr)
                        for dc in range(0,src.width,CORE):
                            ww=min(CORE,src.width-dc)
                            prob,valid=infer_rect(src,dr,dc,hh,ww)
                            dst.write(to_u16(prob,valid),1,window=Window(dc,dr-row,ww,hh))
                    dst.update_tags(SPEC=key,COMPLETE='YES',ROW=str(row))
                assert verify_stripe(tmp,row,h)
                os.replace(tmp,dest)
                print('SAVED stripe',k,flush=True)
            staging=fullpath.with_suffix('.assembling.tif')
            assert not staging.exists(),('Incomplete assembly; inspect',staging)
            with rasterio.open(staging,'w',**profile) as dst:
                for k,row in enumerate(tqdm(range(0,src.height,STRIPE))):
                    h=min(STRIPE,src.height-row)
                    file=d/f'stripe_{k:05d}.tif'
                    assert verify_stripe(file,row,h)
                    with rasterio.open(file) as tile:
                        for rr in range(0,h,256):
                            hh=min(256,h-rr)
                            dst.write(tile.read(1,window=Window(0,rr,src.width,hh)),1,
                                      window=Window(0,row+rr,src.width,hh))
                dst.update_tags(SPEC=key,COMPLETE='YES',PROBABILITY_SCALE=str(SCALE))
                factors=[a for a in (2,4,8,16,32,64) if min(src.width,src.height)//a>=64]
                if factors:dst.build_overviews(factors,Resampling.average)
            os.replace(staging,fullpath)
            print('FULL LUKI MAP SAVED:',fullpath)

## 07 — Optional Nadine full-mosaic annotation comparison

`Class_ID=1` polygons are positive Musanga identifications across the orthomosaic. Use these for map overlays and positive-area diagnostic summaries. **Do not report full-mosaic F1, recall or MAE as independently validated unless completeness of the Nadine inventory is demonstrated.**

In [ ]:
fullpath=OUT/'06_full_mosaic/Luki_M2_warmstart_epoch02_probability_u16_FULL.tif'
if not fullpath.is_file():
    print('Full map not available; run optional 06 for Nadine spatial diagnostics.')
else:
    rows=[]
    with rasterio.open(fullpath) as ds:
        assert ds.crs==CRS
        for i,geometry in enumerate(nad_pos.geometry):
            w=from_bounds(*geometry.bounds,transform=ds.transform)
            col=max(0,int(math.floor(w.col_off)));row=max(0,int(math.floor(w.row_off)))
            right=min(ds.width,int(math.ceil(w.col_off+w.width)))
            bottom=min(ds.height,int(math.ceil(w.row_off+w.height)))
            if right<=col or bottom<=row:continue
            win=Window(col,row,right-col,bottom-row)
            crown_mask=burn_geoms([geometry],win,ds)
            values=ds.read(1,window=win)[crown_mask]
            values=values[values!=NODATA]
            if not len(values):continue
            p=values/SCALE
            rows.append(dict(annotation_row=i,n_valid_pixels=len(values),
                 mean_probability=float(p.mean()),fraction_ge_05=float((p>=.5).mean())))
    nadine_summary=pd.DataFrame(rows)
    display(nadine_summary.describe().round(3))
    nadine_summary.to_csv(OUT/'07_Nadine_positive_crown_diagnostics.csv',index=False)
    print('Nadine-only positive polygon comparison saved. Not an exhaustive full-mosaic validation.')